# Judge agreement (Cohen's kappa)

Compares the H1 judge's verdicts with a second rater's labels on the same turns, and
lists every turn where they disagree so it can be checked by hand.

- **Label sheets:** CSVs with `trial_id`, `context_condition`, `round`, `metric`, and a
  label column (`PASS`, `FAIL`, `NA`, or `EXCLUDE`). Several sheets can be loaded at
  once. The current sheets were labelled by Claude, not a human, and not blind to some
  judge verdicts, so this measures agreement with a second LLM rater, not validation.
- **Kappa** is agreement beyond chance: `(p_o - p_e) / (1 - p_e)`. `NA` counts as a
  third category, so disagreements about whether a metric applies at all are included.
  Read kappa together with raw agreement: it is unstable when one label dominates.
- Rows labelled `EXCLUDE` are skipped (for example `t11` R6 when that run's R3 gave no
  specific answer).

Run All after changing the settings below.

In [22]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent  # this notebook lives in notebooks/
LABEL_DIR = REPO_ROOT / "data" / "research_annotations" / "judge_validation"
# One or more label sheets. full_labels_claude_p*.csv hold every turn and all three
# metrics; second_rater_claude_v1.csv is the earlier stress/probe-only sheet.
LABEL_FILES = sorted(LABEL_DIR.glob("full_labels_claude_p*.csv"))
LABEL_COLUMN = "second_rater_label"
H1_PATH = REPO_ROOT / "output" / "evaluation" / "h1_consistency.jsonl"
EXPORT_NAME = "full_labels_claude"  # prefix for the disagreement export
[p.name for p in LABEL_FILES]

['full_labels_claude_p1.csv',
 'full_labels_claude_p2.csv',
 'full_labels_claude_p3.csv',
 'full_labels_claude_p4.csv',
 'full_labels_claude_p5.csv']

## Load and join

In [23]:
def load_judge(path):
    """Judge rows keyed by (trial_id, condition, round). An incomplete last line (the
    file is still being written by a running evaluation) is skipped and counted."""
    rows, skipped = {}, 0
    with open(path, encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                skipped += 1
                continue
            rows[(row["trial_id"], row["context_condition"], row["round"])] = row
    return rows, skipped


judge_rows, skipped_lines = load_judge(H1_PATH)
# keep_default_na=False keeps the label text "NA" as a label instead of a missing value
labels = pd.concat([pd.read_csv(path, keep_default_na=False) for path in LABEL_FILES], ignore_index=True)
labels["label"] = labels[LABEL_COLUMN].str.strip().str.upper()

records = []
for _, row in labels.iterrows():
    key = (row["trial_id"], row["context_condition"], int(row["round"]))
    judge = judge_rows.get(key)
    entry = judge["decisions"][row["metric"]] if judge else None
    if entry is None:
        judge_label, judge_evidence = "MISSING", None
    elif row["metric"] == "world_consistency":
        judge_label = entry["decision"]
        judge_evidence = "; ".join(
            f"{'OK' if c['consistent'] else 'CONFLICT'}: {c['claim']} (vs {c['checked_against']})"
            for c in entry.get("claims", [])
        )
    else:
        judge_label, judge_evidence = entry["decision"], entry.get("evidence")
    records.append({
        "pilot": row["pilot_id"],
        "trial": row["trial_id"].rsplit("_", 1)[-1],
        "condition": row["context_condition"],
        "round": int(row["round"]),
        "trial_type": row["trial_type"],
        "metric": row["metric"],
        "player_line": row["player_line"],
        "npc_reply": row["npc_reply"],
        "rater": row["label"],
        "rater_reason": row.get("rationale", ""),
        "judge": judge_label,
        "judge_evidence": judge_evidence,
    })
joined = pd.DataFrame(records)
scored = joined[(joined["rater"] != "EXCLUDE") & (joined["judge"] != "MISSING")]

print(f"Label rows: {len(joined)} from {len(LABEL_FILES)} sheet(s)")
print(f"Incomplete judge lines skipped: {skipped_lines}")
print(f"Excluded by the rater: {(joined['rater'] == 'EXCLUDE').sum()}   "
      f"Missing judge rows: {(joined['judge'] == 'MISSING').sum()}")
print(f"Scored label pairs: {len(scored)}")

Label rows: 900 from 5 sheet(s)
Incomplete judge lines skipped: 0
Excluded by the rater: 4   Missing judge rows: 0
Scored label pairs: 896


## Agreement and kappa

Each row shows how many label pairs it covers, raw agreement, kappa, and each side's mix
of labels. When one side used a single label for every turn in a group (for example
the rater marking every `t11` World Consistency turn PASS), chance agreement equals
observed agreement, so kappa is 0 or undefined whatever the agreement: the `note`
column flags these rows. For them, read `agreement` and the disagreement list below
instead of kappa.


In [24]:
def label_mix(counts):
    return " ".join(f"{label}:{counts[label]}" for label in ("PASS", "FAIL", "NA") if counts[label])


def kappa(frame):
    """Agreement, Cohen's kappa, each side's label mix, and a note when kappa cannot
    be read: if either side used a single label, chance agreement equals observed
    agreement and kappa is 0 (or undefined) by construction, whatever the agreement."""
    pairs = list(zip(frame["rater"], frame["judge"]))
    n = len(pairs)
    if n == 0:
        return {"n": 0, "agreement": None, "kappa": None, "rater": "", "judge": "", "note": ""}
    observed = sum(a == b for a, b in pairs) / n
    rater_counts, judge_counts = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    expected = sum(rater_counts[l] * judge_counts[l] for l in rater_counts) / (n * n)
    value = None if expected == 1 else (observed - expected) / (1 - expected)
    note = ""
    if len(rater_counts) == 1 and rater_counts.keys() == judge_counts.keys():
        only = next(iter(rater_counts))
        note = f"both sides gave only {only}" + (": metric does not apply here" if only == "NA" else "")
    elif len(rater_counts) == 1 or len(judge_counts) == 1:
        side = "rater" if len(rater_counts) == 1 else "judge"
        note = f"{side} used one label only: kappa not meaningful, read agreement"
    return {
        "n": n,
        "agreement": round(observed, 2),
        "kappa": None if value is None else round(value, 2),
        "rater": label_mix(rater_counts),
        "judge": label_mix(judge_counts),
        "note": note,
    }


groups = {"all": scored}
for metric in ("context_use", "world_consistency", "no_contradiction"):
    groups[metric] = scored[scored["metric"] == metric]
for trial_type in sorted(scored["trial_type"].unique()):
    groups[f"type {trial_type}"] = scored[scored["trial_type"] == trial_type]
for condition in ("U", "S", "P"):
    groups[f"condition {condition}"] = scored[scored["condition"] == condition]
for pilot in sorted(scored["pilot"].unique()):
    groups[pilot] = scored[scored["pilot"] == pilot]

summary = pd.DataFrame({name: kappa(frame) for name, frame in groups.items()}).T
summary

,n,agreement,kappa,rater,judge,note
all,896,0.91,0.82,PASS:570 FAIL:161 NA:165,PASS:620 FAIL:111 NA:165,
context_use,298,0.81,0.57,PASS:183 FAIL:115,PASS:229 FAIL:69,
world_consistency,300,0.95,0.69,PASS:272 FAIL:28,PASS:270 FAIL:30,
no_contradiction,298,0.97,0.94,PASS:115 FAIL:18 NA:165,PASS:121 FAIL:12 NA:165,
type 1,270,0.93,0.86,PASS:155 FAIL:25 NA:90,PASS:171 FAIL:9 NA:90,
type 2,180,0.86,0.79,PASS:47 FAIL:73 NA:60,PASS:73 FAIL:47 NA:60,
type 3,446,0.92,0.72,PASS:368 FAIL:63 NA:15,PASS:376 FAIL:55 NA:15,
condition U,300,0.9,0.82,PASS:174 FAIL:71 NA:55,PASS:194 FAIL:51 NA:55,
condition S,296,0.9,0.79,PASS:188 FAIL:53 NA:55,PASS:205 FAIL:36 NA:55,
condition P,300,0.93,0.84,PASS:208 FAIL:37 NA:55,PASS:221 FAIL:24 NA:55,


### Kappa per metric and trial type

In [25]:
pd.DataFrame({
    (metric, f"type {t}"): kappa(scored[(scored["metric"] == metric) & (scored["trial_type"] == t)])
    for metric in ("context_use", "world_consistency", "no_contradiction")
    for t in sorted(scored["trial_type"].unique())
}).T

n agreement kappa                   rater                   judge                                                             note
context_use       type 1   90      0.81  0.34         PASS:67 FAIL:23          PASS:84 FAIL:6                                                                 
                  type 2   60      0.67  0.37         PASS:13 FAIL:47         PASS:33 FAIL:27                                                                 
                  type 3  148      0.87  0.68        PASS:103 FAIL:45        PASS:112 FAIL:36                                                                 
world_consistency type 1   90      0.97  0.38          PASS:88 FAIL:2          PASS:87 FAIL:3                                                                 
                  type 2   60       0.9  0.79         PASS:34 FAIL:26         PASS:40 FAIL:20                                                                 
                  type 3  150      0.95   0.0                PASS:150         PASS:143 FAIL:7  rater used one label only: kappa not meaningful, read agreement
no_contradiction  type 1   90       1.0  None                   NA:90                   NA:90              both sides gave only NA: metric does not apply here
                  type 2   60       1.0  None                   NA:60                   NA:60              both sides gave only NA: metric does not apply here
                  type 3  148      0.93   0.8  PASS:115 FAIL:18 NA:15  PASS:121 FAIL:12 NA:15

### Confusion tables (rows = rater, columns = judge)

In [26]:
for metric in ("context_use", "world_consistency", "no_contradiction"):
    subset = scored[scored["metric"] == metric]
    print(metric)
    display(pd.crosstab(subset["rater"], subset["judge"]))

context_use


judge,FAIL,PASS
rater,,
FAIL,64,51
PASS,5,178


world_consistency


judge,FAIL,PASS
rater,,
FAIL,21,7
PASS,9,263


no_contradiction


judge,FAIL,NA,PASS
rater,,,
FAIL,10,0,8
NA,0,165,0
PASS,2,0,113


## Disagreements to confirm

Every label pair where the rater and the judge disagree, with the full dialogue, the
rater's reason, and the judge's own evidence (for World Consistency, its claim list).
The export writes these rows to a CSV with empty `researcher_verdict` and
`researcher_note` columns, so each case can be settled by hand.

In [27]:
disagree = scored[scored["rater"] != scored["judge"]].sort_values(["pilot", "trial", "condition", "round", "metric"])
print(f"{len(disagree)} disagreements")
display(disagree[["pilot", "trial", "condition", "round", "metric", "player_line", "npc_reply",
                  "rater", "rater_reason", "judge", "judge_evidence"]])

82 disagreements


,pilot,trial,condition,round,metric,player_line,npc_reply,rater,rater_reason,judge,judge_evidence
186,cb_hp_012_newt_mcgonagall,t01,P,1,context_use,"Remind me, Professor, where are we, and what time of day is it?","Mr. Scamander, I hardly think it necessary to remind you that we are in the Great Hall at Hogwarts, and it is well past the time for sensible behaviour. Your penchant for chaos is becoming rather tiresome, and I assure you, this 'accident' will not go unaddressed. Now, kindly explain how you intend to rectify this situation before I am forced to take further action.",FAIL,partial: 'in the Great Hall at Hogwarts' but no time of day,PASS,"Elicited fact: Location = The Great Hall, Time = Late Morning. Response: 'we are in the Great Hall at Hogwarts, and it is well past the time for sensible behaviour.'"
180,cb_hp_012_newt_mcgonagall,t01,U,1,context_use,"Remind me, Professor, where are we, and what time of day is it?","Mr. Scamander, while I appreciate your apparent confusion, I assure you that we are in the Great Hall, and it is quite past the time for your whimsical antics. Now, if you would kindly explain how a Niffler came to be causing chaos during breakfast, I would be most obliged.",FAIL,partial: 'we are in the Great Hall' but no time of day,PASS,"Elicited fact: Location = The Great Hall, Time = Late Morning. Response: 'we are in the Great Hall, and it is quite past the time for your whimsical antics.'"
195,cb_hp_012_newt_mcgonagall,t02,P,1,context_use,"Where are you standing, Professor? I can't quite see you from here.","Newt Scamander, I assure you, it is not my visibility that is in question here, but rather your complete disregard for the rules of this school! A Niffler, of all creatures, causing chaos in the Great Hall is not merely an accident; it is a serious breach of responsibility. Now, kindly refrain from further distractions and explain yourself properly before I decide on an appropriate course of action.",FAIL,dodges: 'it is not my visibility that is in question here',PASS,"Elicited fact: Professor McGonagall is at the front of the Great Hall. Response: 'I assure you, it is not my visibility that is in question here' addresses the player's question about her location."
189,cb_hp_012_newt_mcgonagall,t02,U,1,context_use,"Where are you standing, Professor? I can't quite see you from here.","Mr. Scamander, if you had paid half as much attention to your responsibilities as you do to your clever quips, we might not be in this rather chaotic predicament. Now, kindly step forward so I can address this nonsense properly.",FAIL,dodges: 'kindly step forward',PASS,"Elicited fact: The player's turn asks about Professor McGonagall's location. Response: 'Now, kindly step forward so I can address this nonsense properly.'"
198,cb_hp_012_newt_mcgonagall,t03,U,1,context_use,"What exactly is going on in here, Professor? How would you describe the hall right now?","Mr. Scamander, this hall is not a menagerie, and your penchant for chaos is becoming rather tiresome. I suggest you take immediate responsibility for your actions, or I shall be forced to impose a rather stern punishment.",FAIL,dodges: 'this hall is not a menagerie'; never describes it,PASS,"Elicited fact: Location = The Great Hall, Hogwarts. Response: 'this hall is not a menagerie' addresses the location."
...,...,...,...,...,...,...,...,...,...,...,...
615,cb_hp_044_harry_hagrid,t09,S,1,context_use,It's just the two of us in the hut right now. Correct me if I'm wrong.,"Aye, Harry, it is just us fer now. But I can’t help but feel a bit lost without me friends 'round to help with Aragog. He’s been like a brother to me, y’know? I just wish there was summat more I could do fer him... *looks down, wiping a tear from his eye* I just want 'im to be alright.",FAIL,"'Aye, Harry, it is just us fer now'",PASS,"Elicited fact: Location = Hagrid's Hut, Hogwarts. Response: 'Aye, Harry, it is just us fer now.'"
666,cb_hp_044_harry_hagrid,t11,S,3,context_use,"What did you pu

In [28]:
export_path = LABEL_DIR / f"{EXPORT_NAME}_disagreements.csv"
disagree.assign(researcher_verdict="", researcher_note="").to_csv(export_path, index=False)
print("Wrote", export_path)

Wrote /Users/macbookproretina/Downloads/Mycomputer/Usyd/Yr4 2026/thesis/main-research/data/research_annotations/judge_validation/full_labels_claude_disagreements.csv
